<a href="https://colab.research.google.com/github/lbruner954/bruner-python-security-project/blob/main/LogParser.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import re
import json
import csv
import ipaddress
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from datetime import datetime


# ============================================================
# 1. REGEX PATTERNS
# ============================================================

# Apache/Nginx Combined Log Format
# Example:
# 203.0.113.50 - - [13/Sep/2026:17:00:01 -0400] "GET / HTTP/1.1" 200 1024 "-" "Mozilla/5.0"
WEB_LOG_RE = re.compile(
    r'^(?P<ip>\S+)\s+\S+\s+\S+\s+'
    r'\[(?P<timestamp>[^\]]+)\]\s+'
    r'"(?P<method>[A-Z]+)\s+(?P<path>\S+)\s+(?P<protocol>[^"]+)"\s+'
    r'(?P<status>\d{3})\s+'
    r'(?P<bytes>\S+)'
    r'(?:\s+"(?P<referer>[^"]*)"\s+"(?P<user_agent>[^"]*)")?$'
)

# Standard Linux/Unix syslog format
# Example:
# Sep 13 17:20:01 server sshd[1501]: Failed password for root from 198.51.100.44 port 50122 ssh2
SYSLOG_RE = re.compile(
    r'^(?P<timestamp>[A-Z][a-z]{2}\s+\d{1,2}\s+\d{2}:\d{2}:\d{2})\s+'
    r'(?P<host>\S+)\s+'
    r'(?P<program>[\w./-]+)(?:\[(?P<pid>\d+)\])?:\s*'
    r'(?P<message>.*)$'
)

# Failed SSH authentication attempts
SSH_FAILED_LOGIN_RE = re.compile(
    r'(?:Failed password|authentication failure|Invalid user).*?'
    r'(?:for\s+(?:invalid user\s+)?(?P<user>[\w.@$-]+))?.*?'
    r'(?:from|rhost=)(?P<ip>\d{1,3}(?:\.\d{1,3}){3}|[0-9a-fA-F:]+)',
    re.IGNORECASE
)

# Successful SSH authentication attempts
SSH_SUCCESS_LOGIN_RE = re.compile(
    r'Accepted\s+\S+\s+for\s+(?P<user>[\w.@$-]+)\s+from\s+'
    r'(?P<ip>\d{1,3}(?:\.\d{1,3}){3}|[0-9a-fA-F:]+)',
    re.IGNORECASE
)

# Windows Event Log text exports
WINDOWS_EVENT_ID_RE = re.compile(
    r'(?:Event\s*ID|EventID)\s*[:=]\s*(?P<event_id>\d+)',
    re.IGNORECASE
)

WINDOWS_IP_RE = re.compile(
    r'(?:Source\s+Network\s+Address|IpAddress|Client\s+Address)\s*[:=]\s*(?P<ip>\S+)',
    re.IGNORECASE
)

WINDOWS_USER_RE = re.compile(
    r'(?:Account\s+Name|TargetUserName|Username|User)\s*[:=]\s*(?P<user>[^\r\n]+)',
    re.IGNORECASE
)

# Windows Event XML exports
WINDOWS_XML_EVENT_ID_RE = re.compile(
    r'<EventID[^>]*>(?P<event_id>\d+)</EventID>',
    re.IGNORECASE
)

WINDOWS_XML_IP_RE = re.compile(
    r'<Data\s+Name=["\']IpAddress["\']>(?P<ip>[^<]+)</Data>',
    re.IGNORECASE
)

WINDOWS_XML_USER_RE = re.compile(
    r'<Data\s+Name=["\']TargetUserName["\']>(?P<user>[^<]+)</Data>',
    re.IGNORECASE
)

# Suspicious paths commonly used in scanning or exploitation attempts
SUSPICIOUS_PATH_RE = re.compile(
    r'(?i)('
    r'/wp-admin|/wp-login|/xmlrpc\.php|'
    r'/\.env|/\.git|/\.svn|'
    r'/phpmyadmin|/admin|/administrator|'
    r'/cgi-bin|/shell|/cmd|'
    r'\.\./|%2e%2e|'
    r'/etc/passwd|/proc/self/environ|'
    r'/login|/signin'
    r')'
)


# ============================================================
# 2. DATA MODELS
# ============================================================

@dataclass
class LogEvent:
    source_type: str
    timestamp: str | None
    ip: str | None
    user: str | None
    event_type: str
    status_code: int | None
    method: str | None
    path: str | None
    raw: str


@dataclass
class Alert:
    severity: str
    category: str
    message: str
    evidence: str


# ============================================================
# 3. HELPER FUNCTIONS
# ============================================================

def is_valid_ip(value):
    """Return True when value is a valid IPv4 or IPv6 address."""
    if not value or value in {"-", "N/A", "None", "::1"}:
        return False

    try:
        ipaddress.ip_address(value)
        return True
    except ValueError:
        return False


def is_private_or_internal_ip(ip_value):
    """Identify private/internal, loopback, reserved, and link-local addresses."""
    try:
        ip_obj = ipaddress.ip_address(ip_value)
        return (
            ip_obj.is_private
            or ip_obj.is_loopback
            or ip_obj.is_link_local
            or ip_obj.is_reserved
        )
    except ValueError:
        return False


def parse_allowlist(ip_ranges):
    """Convert an IP/CIDR list into ipaddress network objects."""
    networks = []

    for item in ip_ranges:
        try:
            if "/" in item:
                networks.append(ipaddress.ip_network(item, strict=False))
            else:
                address = ipaddress.ip_address(item)
                suffix = "/32" if address.version == 4 else "/128"
                networks.append(ipaddress.ip_network(f"{item}{suffix}", strict=False))
        except ValueError:
            print(f"Warning: Invalid allowlist entry ignored: {item}")

    return networks


def is_allowlisted(ip_value, allowlist):
    """Return True if IP appears inside a configured allowlisted range."""
    try:
        address = ipaddress.ip_address(ip_value)
        return any(address in network for network in allowlist)
    except ValueError:
        return False


def parse_web_timestamp(value):
    """Convert Apache/Nginx time into ISO 8601 format if possible."""
    try:
        return datetime.strptime(value, "%d/%b/%Y:%H:%M:%S %z").isoformat()
    except ValueError:
        return value


# ============================================================
# 4. PARSER FUNCTIONS
# ============================================================

def parse_web_log(line):
    """Parse Apache/Nginx Common or Combined log records."""
    match = WEB_LOG_RE.match(line)

    if not match:
        return None

    data = match.groupdict()
    status = int(data["status"])

    if status in {401, 403}:
        event_type = "web_auth_failure"
    elif status >= 500:
        event_type = "web_server_error"
    elif status >= 400:
        event_type = "web_client_error"
    else:
        event_type = "web_request"

    return LogEvent(
        source_type="apache_nginx",
        timestamp=parse_web_timestamp(data["timestamp"]),
        ip=data["ip"] if is_valid_ip(data["ip"]) else None,
        user=None,
        event_type=event_type,
        status_code=status,
        method=data["method"],
        path=data["path"],
        raw=line.rstrip()
    )


def parse_syslog(line, assumed_year=None):
    """Parse Linux/Unix syslog messages and identify SSH authentication events."""
    if assumed_year is None:
        assumed_year = datetime.now().year

    match = SYSLOG_RE.match(line)

    if not match:
        return None

    data = match.groupdict()
    message = data["message"]

    try:
        timestamp = datetime.strptime(
            f"{assumed_year} {data['timestamp']}",
            "%Y %b %d %H:%M:%S"
        ).isoformat()
    except ValueError:
        timestamp = data["timestamp"]

    failed_match = SSH_FAILED_LOGIN_RE.search(message)
    success_match = SSH_SUCCESS_LOGIN_RE.search(message)

    if failed_match:
        return LogEvent(
            source_type="syslog",
            timestamp=timestamp,
            ip=failed_match.group("ip"),
            user=failed_match.group("user"),
            event_type="failed_login",
            status_code=None,
            method=None,
            path=None,
            raw=line.rstrip()
        )

    if success_match:
        return LogEvent(
            source_type="syslog",
            timestamp=timestamp,
            ip=success_match.group("ip"),
            user=success_match.group("user"),
            event_type="successful_login",
            status_code=None,
            method=None,
            path=None,
            raw=line.rstrip()
        )

    return LogEvent(
        source_type="syslog",
        timestamp=timestamp,
        ip=None,
        user=None,
        event_type="syslog_message",
        status_code=None,
        method=None,
        path=None,
        raw=line.rstrip()
    )


def parse_windows_text(line):
    """
    Parse a single Windows Event Log text-export record.

    Example:
    Event ID: 4625 Account Name: Administrator Source Network Address: 203.0.113.77
    """
    event_match = WINDOWS_EVENT_ID_RE.search(line)

    if not event_match:
        return None

    event_id = event_match.group("event_id")
    ip_match = WINDOWS_IP_RE.search(line)
    user_match = WINDOWS_USER_RE.search(line)

    ip = ip_match.group("ip") if ip_match else None
    user = user_match.group("user").strip() if user_match else None

    if event_id == "4625":
        event_type = "failed_login"
    elif event_id in {"4624", "4648"}:
        event_type = "successful_login"
    else:
        event_type = f"windows_event_{event_id}"

    return LogEvent(
        source_type="windows_event",
        timestamp=None,
        ip=ip if is_valid_ip(ip) else None,
        user=user,
        event_type=event_type,
        status_code=None,
        method=None,
        path=None,
        raw=line.rstrip()
    )


def parse_windows_xml_block(block):
    """Parse one exported Windows Event XML block."""
    event_match = WINDOWS_XML_EVENT_ID_RE.search(block)

    if not event_match:
        return None

    event_id = event_match.group("event_id")
    ip_match = WINDOWS_XML_IP_RE.search(block)
    user_match = WINDOWS_XML_USER_RE.search(block)

    ip = ip_match.group("ip").strip() if ip_match else None
    user = user_match.group("user").strip() if user_match else None

    if event_id == "4625":
        event_type = "failed_login"
    elif event_id in {"4624", "4648"}:
        event_type = "successful_login"
    else:
        event_type = f"windows_event_{event_id}"

    return LogEvent(
        source_type="windows_event",
        timestamp=None,
        ip=ip if is_valid_ip(ip) else None,
        user=user,
        event_type=event_type,
        status_code=None,
        method=None,
        path=None,
        raw=re.sub(r"\s+", " ", block).strip()
    )


# ============================================================
# 5. LOG-FORMAT DETECTION AND FILE INGESTION
# ============================================================

def detect_format(line):
    """Identify the likely log format from a single record."""
    if WEB_LOG_RE.match(line):
        return "web"

    if SYSLOG_RE.match(line):
        return "syslog"

    if WINDOWS_EVENT_ID_RE.search(line):
        return "windows_text"

    return "unknown"


def parse_log_file(file_path, log_format="auto"):
    """
    Read and parse a log file.

    Supported formats:
    - auto
    - web
    - syslog
    - windows_text
    - windows_xml
    """
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(
            f"Could not find '{file_path}'. Upload it to Jupyter or verify the filename."
        )

    contents = path.read_text(encoding="utf-8", errors="replace")
    events = []

    # Parse Windows XML blocks.
    if log_format == "windows_xml" or "<Event" in contents:
        xml_blocks = re.findall(
            r"<Event\b.*?</Event>",
            contents,
            re.DOTALL | re.IGNORECASE
        )

        for block in xml_blocks:
            event = parse_windows_xml_block(block)

            if event:
                events.append(event)

        return events

    # Parse line-by-line logs.
    for line in contents.splitlines():
        if not line.strip():
            continue

        selected_format = (
            log_format if log_format != "auto" else detect_format(line)
        )

        if selected_format == "web":
            event = parse_web_log(line)
        elif selected_format == "syslog":
            event = parse_syslog(line)
        elif selected_format == "windows_text":
            event = parse_windows_text(line)
        else:
            event = None

        if event:
            events.append(event)

    return events


# ============================================================
# 6. ANOMALY DETECTION
# ============================================================

def group_by_time_window(events, window_minutes=5):
    """Group timestamped records into fixed time windows."""
    windows = defaultdict(list)

    for event in events:
        if not event.timestamp:
            windows["unknown-time"].append(event)
            continue

        try:
            timestamp = datetime.fromisoformat(event.timestamp)
            bucket_minute = (timestamp.minute // window_minutes) * window_minutes

            window_start = timestamp.replace(
                minute=bucket_minute,
                second=0,
                microsecond=0
            )

            windows[window_start.isoformat()].append(event)

        except ValueError:
            windows["unknown-time"].append(event)

    return windows


def detect_repeated_failed_logins(events, threshold=5):
    """Detect repeated failed logins by IP address or username."""
    alerts = []

    failures = [
        event for event in events
        if event.event_type in {"failed_login", "web_auth_failure"}
    ]

    failures_by_ip = defaultdict(list)
    failures_by_user = defaultdict(list)

    for event in failures:
        if event.ip:
            failures_by_ip[event.ip].append(event)

        if event.user:
            failures_by_user[event.user].append(event)

    for ip, matching_events in failures_by_ip.items():
        if len(matching_events) >= threshold:
            alerts.append(Alert(
                severity="HIGH",
                category="Repeated Failed Logins by IP",
                message=(
                    f"{len(matching_events)} failed authentication attempts "
                    f"were detected from IP address {ip}."
                ),
                evidence="\n".join(event.raw for event in matching_events[:5])
            ))

    for user, matching_events in failures_by_user.items():
        if len(matching_events) >= threshold:
            alerts.append(Alert(
                severity="HIGH",
                category="Repeated Failed Logins by User",
                message=(
                    f"User account '{user}' had {len(matching_events)} "
                    f"failed authentication attempts."
                ),
                evidence="\n".join(event.raw for event in matching_events[:5])
            ))

    return alerts


def detect_password_spraying(events, distinct_user_threshold=3):
    """Detect one source IP failing against several different accounts."""
    alerts = []
    users_by_ip = defaultdict(set)

    for event in events:
        if event.event_type == "failed_login" and event.ip and event.user:
            users_by_ip[event.ip].add(event.user)

    for ip, users in users_by_ip.items():
        if len(users) >= distinct_user_threshold:
            alerts.append(Alert(
                severity="HIGH",
                category="Possible Password Spraying",
                message=(
                    f"IP address {ip} made failed authentication attempts "
                    f"against {len(users)} different accounts."
                ),
                evidence=f"Targeted accounts: {', '.join(sorted(users))}"
            ))

    return alerts


def detect_unusual_ips(events, allowlist):
    """
    Detect public IP addresses that are not present in an approved allowlist.

    Note: This is a simple lab-rule baseline. A real SOC would normally
    combine an allowlist with historical behavior, GeoIP, ASN, and threat intel.
    """
    alerts = []
    checked_ips = set()

    for event in events:
        if not event.ip or event.ip in checked_ips:
            continue

        checked_ips.add(event.ip)

        if (
            is_valid_ip(event.ip)
            and not is_private_or_internal_ip(event.ip)
            and not is_allowlisted(event.ip, allowlist)
        ):
            alerts.append(Alert(
                severity="MEDIUM",
                category="Unusual External IP",
                message=(
                    f"Public IP address {event.ip} was observed but is not "
                    f"in the configured allowlist."
                ),
                evidence=event.raw
            ))

    return alerts


def detect_traffic_spikes(events, window_minutes=5, threshold=100):
    """Detect high request volume within a fixed time window."""
    alerts = []

    web_events = [
        event for event in events
        if event.source_type == "apache_nginx"
    ]

    grouped_events = group_by_time_window(web_events, window_minutes)

    for time_window, matching_events in grouped_events.items():
        if time_window == "unknown-time":
            continue

        if len(matching_events) >= threshold:
            top_ips = Counter(
                event.ip for event in matching_events if event.ip
            ).most_common(3)

            top_ip_text = ", ".join(
                f"{ip} ({count} requests)"
                for ip, count in top_ips
            )

            alerts.append(Alert(
                severity="HIGH",
                category="Traffic Spike",
                message=(
                    f"{len(matching_events)} web requests occurred during the "
                    f"{window_minutes}-minute window starting at {time_window}."
                ),
                evidence=(
                    f"Most active source IPs: "
                    f"{top_ip_text if top_ip_text else 'No IP data available'}"
                )
            ))

    return alerts


def detect_http_error_spikes(events, threshold=10):
    """Detect source IPs that create excessive HTTP 4xx or 5xx errors."""
    alerts = []

    http_errors = [
        event for event in events
        if event.status_code is not None and event.status_code >= 400
    ]

    errors_by_ip = defaultdict(list)

    for event in http_errors:
        if event.ip:
            errors_by_ip[event.ip].append(event)

    for ip, matching_events in errors_by_ip.items():
        if len(matching_events) >= threshold:
            alerts.append(Alert(
                severity="MEDIUM",
                category="Excessive HTTP Errors",
                message=(
                    f"IP address {ip} produced {len(matching_events)} "
                    f"HTTP 4xx/5xx responses."
                ),
                evidence="\n".join(event.raw for event in matching_events[:5])
            ))

    return alerts


def detect_suspicious_paths(events, threshold=3):
    """Detect repeated access to suspicious web paths."""
    alerts = []
    suspicious_requests = defaultdict(list)

    for event in events:
        if event.path and SUSPICIOUS_PATH_RE.search(event.path):
            key = (event.ip or "unknown-ip", event.path)
            suspicious_requests[key].append(event)

    for (ip, path), matching_events in suspicious_requests.items():
        if len(matching_events) >= threshold:
            alerts.append(Alert(
                severity="HIGH",
                category="Suspicious Web Probing",
                message=(
                    f"IP address {ip} requested suspicious path '{path}' "
                    f"{len(matching_events)} times."
                ),
                evidence="\n".join(event.raw for event in matching_events[:5])
            ))

    return alerts


def run_detection_pipeline(
    events,
    allowlist,
    failed_login_threshold=5,
    traffic_spike_threshold=100,
    http_error_threshold=10,
    suspicious_path_threshold=3,
    password_spray_user_threshold=3,
    time_window_minutes=5
):
    """Run every threshold-based detection rule and sort alerts by severity."""
    alerts = []

    alerts.extend(
        detect_repeated_failed_logins(
            events,
            threshold=failed_login_threshold
        )
    )

    alerts.extend(
        detect_password_spraying(
            events,
            distinct_user_threshold=password_spray_user_threshold
        )
    )

    alerts.extend(
        detect_unusual_ips(events, allowlist)
    )

    alerts.extend(
        detect_traffic_spikes(
            events,
            window_minutes=time_window_minutes,
            threshold=traffic_spike_threshold
        )
    )

    alerts.extend(
        detect_http_error_spikes(
            events,
            threshold=http_error_threshold
        )
    )

    alerts.extend(
        detect_suspicious_paths(
            events,
            threshold=suspicious_path_threshold
        )
    )

    severity_order = {
        "CRITICAL": 0,
        "HIGH": 1,
        "MEDIUM": 2,
        "LOW": 3
    }

    return sorted(
        alerts,
        key=lambda alert: severity_order.get(alert.severity, 99)
    )


# ============================================================
# 7. REPORTING
# ============================================================

def calculate_statistics(events):
    """Create event totals used in the report."""
    return {
        "total_events": len(events),
        "events_by_source": Counter(event.source_type for event in events),
        "events_by_type": Counter(event.event_type for event in events),
        "unique_ips": sorted({event.ip for event in events if event.ip}),
        "http_statuses": Counter(
            event.status_code
            for event in events
            if event.status_code is not None
        )
    }


def write_markdown_report(input_file, events, alerts, output_file):
    """Create a Markdown security analysis report."""
    stats = calculate_statistics(events)
    severity_counts = Counter(alert.severity for alert in alerts)

    with open(output_file, "w", encoding="utf-8") as report:
        report.write("# Security Log Analysis Report\n\n")
        report.write(
            f"**Report generated:** "
            f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
        )
        report.write(f"**Input file:** `{input_file}`\n\n")

        report.write("## Executive Summary\n\n")
        report.write(f"- Total normalized log events: {stats['total_events']}\n")
        report.write(f"- Unique source IP addresses: {len(stats['unique_ips'])}\n")
        report.write(f"- Total generated alerts: {len(alerts)}\n")
        report.write(f"- High-severity alerts: {severity_counts['HIGH']}\n")
        report.write(f"- Medium-severity alerts: {severity_counts['MEDIUM']}\n\n")

        report.write("## Parsing Summary\n\n")
        report.write("### Events by Log Source\n\n")

        for source, count in stats["events_by_source"].items():
            report.write(f"- {source}: {count}\n")

        report.write("\n### Events by Type\n\n")

        for event_type, count in stats["events_by_type"].items():
            report.write(f"- {event_type}: {count}\n")

        if stats["http_statuses"]:
            report.write("\n### HTTP Status Code Summary\n\n")

            for status, count in sorted(stats["http_statuses"].items()):
                report.write(f"- HTTP {status}: {count}\n")

        report.write("\n## Detected Security Alerts\n\n")

        if not alerts:
            report.write(
                "No configured threshold-based detections generated alerts.\n"
            )
        else:
            for number, alert in enumerate(alerts, start=1):
                report.write(f"### Alert {number}: {alert.category}\n\n")
                report.write(f"- Severity: **{alert.severity}**\n")
                report.write(f"- Description: {alert.message}\n")
                report.write("- Evidence:\n\n")
                report.write("```text\n")
                report.write(alert.evidence + "\n")
                report.write("```\n\n")

        report.write("## Recommended Analyst Actions\n\n")

        if alerts:
            report.write(
                "- Validate whether source IP addresses belong to approved users, "
                "VPN gateways, proxy servers, monitoring tools, or partners.\n"
            )
            report.write(
                "- Investigate repeated failed logins for brute-force activity, "
                "password spraying, stale credentials, or service misconfiguration.\n"
            )
            report.write(
                "- Review suspicious web requests for successful responses, "
                "unexpected application behavior, and possible compromise indicators.\n"
            )
            report.write(
                "- Preserve the original logs before changing firewall rules, "
                "blocking addresses, or resetting user accounts.\n"
            )
        else:
            report.write(
                "- Continue collecting logs and tune detection thresholds based "
                "on normal organizational activity.\n"
            )


def write_json_events(events, output_file):
    """Save normalized parsed events in JSON format."""
    with open(output_file, "w", encoding="utf-8") as handle:
        json.dump(
            [asdict(event) for event in events],
            handle,
            indent=2
        )


def write_csv_events(events, output_file):
    """Save normalized parsed events in CSV format."""
    fields = [
        "source_type",
        "timestamp",
        "ip",
        "user",
        "event_type",
        "status_code",
        "method",
        "path",
        "raw"
    ]

    with open(output_file, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()

        for event in events:
            writer.writerow(asdict(event))


# ============================================================
# 8. NOTEBOOK CONFIGURATION: CHANGE THESE VALUES
# ============================================================

# Enter the exact filename uploaded to the Jupyter Notebook environment.
INPUT_FILE = "access.log"

# Choose: "auto", "web", "syslog", "windows_text", or "windows_xml".
LOG_FORMAT = "web"

# Generated output files.
REPORT_FILE = "security_report.md"
JSON_FILE = "parsed_events.json"
CSV_FILE = "parsed_events.csv"

# Thresholds.
FAILED_LOGIN_THRESHOLD = 5
TRAFFIC_SPIKE_THRESHOLD = 100
HTTP_ERROR_THRESHOLD = 10
SUSPICIOUS_PATH_THRESHOLD = 3
PASSWORD_SPRAY_USER_THRESHOLD = 3
TIME_WINDOW_MINUTES = 5

# Add your expected or trusted networks here.
# An empty list means every public IP is considered unusual.
ALLOWLIST_IPS = [
    "10.0.0.0/8",
    "172.16.0.0/12",
    "192.168.0.0/16"
]


# ============================================================
# 9. RUN THE ANALYSIS
# ============================================================

try:
    allowlist = parse_allowlist(ALLOWLIST_IPS)

    events = parse_log_file(
        file_path=INPUT_FILE,
        log_format=LOG_FORMAT
    )

    alerts = run_detection_pipeline(
        events=events,
        allowlist=allowlist,
        failed_login_threshold=FAILED_LOGIN_THRESHOLD,
        traffic_spike_threshold=TRAFFIC_SPIKE_THRESHOLD,
        http_error_threshold=HTTP_ERROR_THRESHOLD,
        suspicious_path_threshold=SUSPICIOUS_PATH_THRESHOLD,
        password_spray_user_threshold=PASSWORD_SPRAY_USER_THRESHOLD,
        time_window_minutes=TIME_WINDOW_MINUTES
    )

    write_markdown_report(
        input_file=INPUT_FILE,
        events=events,
        alerts=alerts,
        output_file=REPORT_FILE
    )

    write_json_events(events, JSON_FILE)
    write_csv_events(events, CSV_FILE)

    print("Security log analysis complete.")
    print(f"Parsed events: {len(events)}")
    print(f"Generated alerts: {len(alerts)}")
    print(f"Markdown report created: {REPORT_FILE}")
    print(f"JSON event file created: {JSON_FILE}")
    print(f"CSV event file created: {CSV_FILE}")

    if alerts:
        print("\nAlerts:")
        for number, alert in enumerate(alerts, start=1):
            print(
                f"{number}. [{alert.severity}] "
                f"{alert.category}: {alert.message}"
            )
    else:
        print("\nNo alerts met the current thresholds.")

except FileNotFoundError as error:
    print(error)
    print("\nFiles visible to this notebook:")

    for item in Path(".").iterdir():
        print("-", item.name)

Could not find 'access.log'. Upload it to Jupyter or verify the filename.

Files visible to this notebook:
- .config
- sample_data
